Camada Gold e perguntas de negócio

Pipeline Medallion (Raw → Silver → Gold) sobre viagens a serviço do Portal da Transparência.

**Pré-requisito:** rodar `0_criar_banco.sql`, `1_extrair.py` e `2_transformar.py` antes deste notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from banco import conectar, executar

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

conexao = conectar()
print("Conectado.")

## Camada Gold

Agrega custo por órgão solicitante (JOIN viagem + pagamento).

In [ ]:
executar(conexao, "DROP TABLE IF EXISTS gold_custo_orgao CASCADE")
executar(conexao, "DROP VIEW IF EXISTS vw_gold_custo_orgao CASCADE")

executar(
    conexao,
    """
    CREATE TABLE gold_custo_orgao AS
    SELECT
        v.nome_orgao_solicitante AS orgao,
        COUNT(DISTINCT v.id_viagem) AS qtd_viagens,
        COALESCE(SUM(p.valor), 0) AS custo_total_pagamento,
        COALESCE(AVG(p.valor), 0) AS ticket_medio_pagamento,
        COALESCE(SUM(v.valor_total), 0) AS custo_total_viagem
    FROM silver_viagem v
    LEFT JOIN silver_pagamento p ON p.id_viagem = v.id_viagem
    WHERE v.nome_orgao_solicitante IS NOT NULL
    GROUP BY v.nome_orgao_solicitante
    """,
)

executar(
    conexao,
    """
    CREATE VIEW vw_gold_custo_orgao AS
    SELECT
        v.nome_orgao_solicitante AS orgao,
        COUNT(DISTINCT v.id_viagem) AS qtd_viagens,
        COALESCE(SUM(p.valor), 0) AS custo_total_pagamento,
        COALESCE(AVG(p.valor), 0) AS ticket_medio_pagamento,
        COALESCE(SUM(v.valor_total), 0) AS custo_total_viagem
    FROM silver_viagem v
    LEFT JOIN silver_pagamento p ON p.id_viagem = v.id_viagem
    WHERE v.nome_orgao_solicitante IS NOT NULL
    GROUP BY v.nome_orgao_solicitante
    """,
)

pd.read_sql("SELECT * FROM gold_custo_orgao ORDER BY custo_total_pagamento DESC LIMIT 5", conexao)

## Pergunta 1 — Quais os 5 órgãos com maior custo total?

In [ ]:
q1 = pd.read_sql(
    """
    SELECT orgao, custo_total_pagamento, qtd_viagens
    FROM gold_custo_orgao
    ORDER BY custo_total_pagamento DESC
    LIMIT 5
    """,
    conexao,
)
display(q1)

fig, ax = plt.subplots()
sns.barplot(data=q1, x="custo_total_pagamento", y="orgao", ax=ax, hue="orgao", legend=True)
ax.set_title("Top 5 órgãos por custo total de pagamento")
ax.set_xlabel("Custo total (R$)")
ax.set_ylabel("Órgão solicitante")
ax.legend(title="Órgão", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Pergunta 2 — Quais os 3 destinos com maior custo médio por viagem?

In [ ]:
q2 = pd.read_sql(
    """
    SELECT
        destinos,
        COUNT(*) AS qtd_viagens,
        ROUND(AVG(valor_total)::numeric, 2) AS custo_medio
    FROM silver_viagem
    WHERE destinos IS NOT NULL AND valor_total IS NOT NULL
    GROUP BY destinos
    HAVING COUNT(*) >= 5
    ORDER BY custo_medio DESC
    LIMIT 3
    """,
    conexao,
)
display(q2)

fig, ax = plt.subplots()
sns.barplot(data=q2, x="custo_medio", y="destinos", ax=ax, hue="destinos", legend=True)
ax.set_title("Top 3 destinos por custo médio por viagem")
ax.set_xlabel("Custo médio (R$)")
ax.set_ylabel("Destino")
ax.legend(title="Destino", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Pergunta 3 — Qual viagem teve maior duração e qual o custo total?

In [ ]:
q3 = pd.read_sql(
    """
    SELECT id_viagem, nome_viajante, destinos, duracao_dias, valor_total,
           nome_orgao_solicitante
    FROM silver_viagem
    WHERE duracao_dias IS NOT NULL
    ORDER BY duracao_dias DESC, valor_total DESC
    LIMIT 10
    """,
    conexao,
)
display(q3)

fig, ax = plt.subplots()
sns.scatterplot(
    data=q3, x="duracao_dias", y="valor_total",
    hue="nome_orgao_solicitante", s=80, ax=ax
)
ax.set_title("Duração x custo — 10 viagens mais longas")
ax.set_xlabel("Duração (dias)")
ax.set_ylabel("Custo total (R$)")
ax.legend(title="Órgão", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Pergunta 4 — Qual tipo de pagamento tem maior valor médio?

In [ ]:
q4 = pd.read_sql(
    """
    SELECT
        tipo_pagamento,
        COUNT(*) AS qtd,
        ROUND(AVG(valor)::numeric, 2) AS valor_medio,
        ROUND(SUM(valor)::numeric, 2) AS valor_total
    FROM silver_pagamento
    GROUP BY tipo_pagamento
    ORDER BY valor_medio DESC
    """,
    conexao,
)
display(q4)

fig, ax = plt.subplots()
sns.barplot(data=q4, x="tipo_pagamento", y="valor_medio", ax=ax, hue="tipo_pagamento", legend=True)
ax.set_title("Valor médio por tipo de pagamento")
ax.set_xlabel("Tipo de pagamento")
ax.set_ylabel("Valor médio (R$)")
ax.legend(title="Tipo", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Pergunta 5 — Qual meio de transporte é mais usado nos trechos?

In [ ]:
q5 = pd.read_sql(
    """
    SELECT meio_transporte, COUNT(*) AS qtd_trechos
    FROM silver_trecho
    WHERE meio_transporte IS NOT NULL
    GROUP BY meio_transporte
    ORDER BY qtd_trechos DESC
    """,
    conexao,
)
display(q5)

fig, ax = plt.subplots()
sns.barplot(data=q5, x="qtd_trechos", y="meio_transporte", ax=ax, hue="meio_transporte", legend=True)
ax.set_title("Meios de transporte mais usados nos trechos")
ax.set_xlabel("Quantidade de trechos")
ax.set_ylabel("Meio de transporte")
ax.legend(title="Meio", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Pergunta 6 — Qual UF de destino aparece com mais frequência nos trechos?

In [ ]:
q6 = pd.read_sql(
    """
    SELECT destino_uf, COUNT(*) AS qtd
    FROM silver_trecho
    WHERE destino_uf IS NOT NULL
      AND LOWER(destino_uf) NOT LIKE 'sem %'
    GROUP BY destino_uf
    ORDER BY qtd DESC
    LIMIT 10
    """,
    conexao,
)
display(q6)

fig, ax = plt.subplots()
sns.barplot(data=q6, x="destino_uf", y="qtd", ax=ax, hue="destino_uf", legend=True)
ax.set_title("Top 10 UFs de destino nos trechos")
ax.set_xlabel("UF de destino")
ax.set_ylabel("Quantidade de trechos")
ax.legend(title="UF", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Insights

- Os órgãos no topo do ranking concentram boa parte do gasto com pagamentos de viagem.
- Destinos com alto custo médio costumam misturar deslocamentos longos e diárias elevadas.
- A duração máxima ajuda a identificar outliers (viagens atípicas / afastamentos longos).
- O tipo de pagamento e o meio de transporte mostram o perfil operacional (passagem aérea vs. outros).
- A UF de destino mais frequente indica para onde o deslocamento federal se concentra no período.

In [ ]:
conexao.close()
print("Conexao encerrada.")